# 多模態 Embedding 與跨模態 RAG

> 模組 05 第 5 節 · 把 02 的純文字 RAG 升級為「圖文混合 + 文件視覺」的跨模態檢索增強生成

## 本 notebook 的一句話定位

**RAG = 檢索（嵌入） + 生成（VLM）**。你在 [`02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb`](../../02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb) 已經把「文字 query → dense retrieval（FAISS）→ cross-encoder rerank → 餵 LLM」這條管線跑完了。本節只改一件事：**把嵌入空間從「純文字」換成「圖文共享空間」**，並把生成端從純文字 LLM 換成 VLM。索引邏輯、normalize、top-k 搜尋的心智模型完全不變——這就是「嵌入即介面」。

## 學習目標

1. 看懂跨模態 RAG 架構，並能逐一對照它與 02 純文字 RAG 的異同。
2. 用 `jinaai/jina-clip-v2` 把圖片與中文文字編碼到**同一個向量空間**，存進 FAISS。
3. 用 `vidore/colpali-v1.3` 對 PDF / 文件截圖做 **late-interaction 視覺檢索**（免 OCR 的 DocQA）。
4. 以文字或圖片當 query，跨模態取回 top-k 證據。
5. 把取回的圖文證據用 `apply_chat_template` 組進 VLM 的多模態 messages，生成**有引用來源**的中文回答。
6. 用 Recall@k / NDCG 評測檢索，並對生成端做基本幻覺檢查。
7. 設計嵌入版本化與增量更新的索引維護策略。

## 前置知識（請先讀完）

| 你需要先會的 | 在哪學的 |
| :--- | :--- |
| FAISS dense retrieval + cross-encoder rerank | [`retrieval_bot.ipynb`](../../02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb) |
| 雙塔對比學習、normalize、cosine 檢索 | [`02-Adv-tasks/04-sentence_similarity/dual_model.ipynb`](../../02-Adv-tasks/04-sentence_similarity/dual_model.ipynb) |
| CLIP / SigLIP 共享嵌入、Recall@k / MRR | [`02-clip_retrieval/clip_image_text_retrieval.ipynb`](../02-clip_retrieval/clip_image_text_retrieval.ipynb) |
| VLM 載入、image token 進 chat 模板 | [`03-vlm_vqa_captioning/vlm_vqa_captioning.ipynb`](../03-vlm_vqa_captioning/vlm_vqa_captioning.ipynb) |
| `BitsAndBytesConfig` 4-bit 量化 | [`04-kbits-tuning/README.md`](../../04-kbits-tuning/README.md) |
| 音訊 → 文字（可選的第四個模態） | [`04-asr_whisper/whisper_asr.ipynb`](../04-asr_whisper/whisper_asr.ipynb) |

> 全 repo 的 2026 慣例聖經：[`../../00-Setup-and-Foundations/01-2026-conventions.md`](../../00-Setup-and-Foundations/01-2026-conventions.md)。

## 0. 版本鎖定與環境

**WHY 把版本鎖在最前面**：跨模態管線同時牽動 transformers（VLM）、faiss（索引）、colpali-engine（late-interaction）三套生態。它們的 API 在 2025→2026 之間都動過，不鎖版本最常見的崩潰是「processor 的 `apply_chat_template` 簽名變了」或「colpali 的 score 函式換名字」。這跟你在 02 只需要 `faiss-cpu` 的單純環境不同——模態變多，依賴面就變大。

下面這格只負責安裝，**不要執行訓練/推論**。

In [ ]:
# Pin versions for a reproducible 2026 multimodal RAG stack.
# Run this once; restart the kernel afterwards if anything was upgraded.
%pip install -q \
    "transformers>=4.46" \
    "datasets>=3.0" \
    "accelerate>=1.0" \
    "bitsandbytes>=0.44" \
    "peft>=0.13" \
    "evaluate>=0.4" \
    "faiss-cpu" \
    "pillow" \
    "pymupdf" \
    "qwen-vl-utils" \
    "colpali-engine>=0.3.5" \
    "einops"

# Optional: only needed if you add the audio modality (Whisper) at the end.
# %pip install -q "soundfile" "librosa"

# jina-clip-v2 ships custom code; transformers will load it via trust_remote_code=True.
# einops is required by that custom modeling file.

In [ ]:
import os
import json
import torch

# Reproducibility: same seed convention as the rest of the repo.
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# bfloat16 on GPU, float32 fallback on CPU (CPU has no bf16 matmul fast path).
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print(f"device={DEVICE}, dtype={DTYPE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. 跨模態 RAG 架構（對照 02 純文字 RAG）

你在 02 跑的是這條管線：

```
text query --(text encoder)--> vector --(FAISS top-k)--> text passages
         --(cross-encoder rerank)--> 重排後的 passages
         --(prompt 模板)--> LLM --> 文字回答
```

本節升級成跨模態，**只有兩個方塊換了內容物，連線完全一樣**：

```
text / image query --(jina-clip-v2,  共享嵌入)--> vector
                   --(FAISS top-k)--> 圖 + 文 混合證據          ← retriever 換成跨模態
                   --(可選 ColPali late-interaction rerank)--> 文件頁
                   --(apply_chat_template, content=[image,text])--> VLM --> 有引用的中文回答  ← generator 換成 VLM
```

### 兩版逐項對照

| 環節 | 02 純文字 RAG | 本節跨模態 RAG | 變了什麼 |
| :--- | :--- | :--- | :--- |
| 語料 | 文字 passages | 圖片 + 中文說明 + PDF 截圖 | 多了非文字模態 |
| 編碼器 | BERT 雙塔（文字塔×2） | jina-clip-v2（影像塔 + 文字塔，共享空間） | 一塔換成影像塔 |
| 索引 | `faiss.IndexFlatIP(768)` | `faiss.IndexFlatIP(dim)` | **完全一樣** |
| normalize | `faiss.normalize_L2` | `faiss.normalize_L2` | **完全一樣** |
| rerank | BERT cross-encoder | ColPali late-interaction（可選） | 概念同：精排 |
| 生成 | 文字 LLM | Qwen2.5-VL（4-bit） | content 變成 list（含 image） |

**核心洞察**：跨模態 RAG 不是新東西，是把「嵌入」這個介面的兩端從同模態擴成異模態。只要 query 與文件最後都變成「同一個空間裡的向量」，下游 FAISS 一個字都不用改。這就是為什麼我們從第一格就反覆強調：**這一步你在單模態版本已經做過**。

## 2. 建立混合語料（圖片 + 中文說明 + 可選 PDF 截圖）

**WHY 先談語料結構**：02 的語料是一張表（id, text）。跨模態語料多了一個維度——每筆證據要記「它是什麼模態」「原始檔在哪」，因為生成端要靠這個資訊把圖片塞回 VLM 並標出引用來源。資料結構錯了，後面引用就一定錯，所以這一步先把結構定死。

我們的語料是一個 `list[dict]`，每筆是一個「文件單元（doc unit）」：

```python
{
  "id": "img_001",
  "modality": "image",          # image | text | pdf_page
  "path": "corpus/cat.jpg",     # 原始檔，VLM 引用時要用
  "caption": "一隻橘貓趴在窗台上曬太陽",  # 中文說明，給人看也給文字檢索備援
}
```

好品味的關鍵：**所有模態共用同一個 schema**。不要為圖片開一個欄位、為文字開另一個欄位再用 if 去分流——那是 02 沒有的特殊情況，會在你加第三種模態時爆炸。

In [ ]:
from pathlib import Path
from PIL import Image

CORPUS_DIR = Path("corpus")
CORPUS_DIR.mkdir(exist_ok=True)

# In a real project you would point these at your own assets. Here we build a tiny
# synthetic corpus so the notebook is self-contained. Replace paths with real files.
#
# Unified schema for EVERY modality -> no per-modality branching downstream.
CORPUS = [
    {"id": "img_001", "modality": "image", "path": "corpus/cat_window.jpg",
     "caption": "一隻橘貓趴在窗台上曬太陽，背景是藍天"},
    {"id": "img_002", "modality": "image", "path": "corpus/red_car.jpg",
     "caption": "一輛紅色跑車停在城市街道旁"},
    {"id": "img_003", "modality": "image", "path": "corpus/mountain.jpg",
     "caption": "清晨的雪山與山下的針葉林"},
    # Pure-text doc units coexist in the SAME index (this is the whole point).
    {"id": "txt_001", "modality": "text", "path": None,
     "caption": "貓科動物喜歡待在溫暖且能觀察四周的高處"},
    {"id": "txt_002", "modality": "text", "path": None,
     "caption": "跑車的低風阻外型有助於高速行駛時的穩定性"},
]

print(f"corpus size = {len(CORPUS)}")
for d in CORPUS:
    print(d["id"], d["modality"], "|", d["caption"])

In [ ]:
# Helper: synthesize placeholder images so the notebook runs end-to-end without
# external downloads. In production, skip this and use your real corpus images.
from PIL import Image, ImageDraw

def _make_placeholder(path: str, color, label: str) -> None:
    p = Path(path)
    if p.exists():
        return
    img = Image.new("RGB", (336, 336), color=color)
    draw = ImageDraw.Draw(img)
    draw.text((20, 160), label, fill=(255, 255, 255))
    img.save(p)

_make_placeholder("corpus/cat_window.jpg", (210, 140, 60), "cat")
_make_placeholder("corpus/red_car.jpg", (180, 40, 40), "car")
_make_placeholder("corpus/mountain.jpg", (90, 120, 160), "mountain")

# Convenience accessor used throughout the notebook.
def load_image(doc: dict) -> Image.Image:
    return Image.open(doc["path"]).convert("RGB")

## 3. 用 jina-clip-v2 產生統一嵌入並存入 FAISS

**WHY jina-clip-v2 而非原版 CLIP**：02 的 `clip_retrieval` 用 SigLIP/CLIP 已經能做圖文檢索，但原版 CLIP 文字塔對中文很弱。`jina-clip-v2` 是 2026 推薦的圖文統一嵌入模型，**多語（含中文）且圖與文輸出同維度、可直接互算 cosine**——這正是 RAG 語料同時含中文 caption 與圖片時的剛需。

關鍵心智模型（和 02 一模一樣）：

1. 把每筆 doc unit 編碼成向量（圖片走影像塔、文字走文字塔，**但落在同一空間**）。
2. `faiss.normalize_L2` 後丟進 `IndexFlatIP`——內積等於 cosine，這是你在 02 寫過的 `faiss.IndexFlatIP(768)`。
3. query 同樣編碼、normalize、`index.search`。

唯一的新概念：**doc 是圖是文，決定走哪個 encode 方法**，但兩者輸出可放進同一個 index。

In [ ]:
from transformers import AutoModel

EMBED_MODEL_ID = "jinaai/jina-clip-v2"

# jina-clip-v2 ships a custom modeling file -> trust_remote_code=True is required.
# It exposes encode_text() / encode_image() that already return L2-normalizable
# vectors in a SHARED space (text dim == image dim). This is what lets one FAISS
# index hold both modalities.
#
# VRAM: ~2-4 GB in bf16. On CPU it works but image encoding is slow.
#
# Lighter alternative for <4 GB GPU or CPU-only:
#   from transformers import AutoModel, AutoProcessor
#   EMBED_MODEL_ID = "google/siglip2-base-patch16-224"  # see 02-clip_retrieval
embed_model = AutoModel.from_pretrained(
    EMBED_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=DTYPE,
).to(DEVICE).eval()

print("embedding model loaded:", EMBED_MODEL_ID)

In [ ]:
import numpy as np

@torch.inference_mode()
def embed_corpus(corpus: list[dict]) -> np.ndarray:
    """Encode every doc unit into the SHARED jina-clip space.

    image docs -> encode_image(PIL.Image)
    text  docs -> encode_text(caption)
    Both branches return vectors of the same dim, so they live in one index.
    """
    vectors = []
    for doc in corpus:
        if doc["modality"] == "image":
            vec = embed_model.encode_image([load_image(doc)])
        else:  # "text" or "pdf_page" caption fallback
            vec = embed_model.encode_text([doc["caption"]])
        # encode_* may return numpy or tensor depending on version; normalize to np.
        vec = np.asarray(vec, dtype="float32")
        vectors.append(vec[0])
    return np.stack(vectors).astype("float32")

corpus_vecs = embed_corpus(CORPUS)
EMB_DIM = corpus_vecs.shape[1]
print("corpus matrix:", corpus_vecs.shape, "| embedding dim:", EMB_DIM)

In [ ]:
import faiss

# Identical to 02's retrieval_bot: inner-product index over L2-normalized vectors
# == cosine similarity. The only difference from 02 is the dim (jina-clip's dim,
# not BERT's 768) and that the rows are a MIX of image and text docs.
index = faiss.IndexFlatIP(EMB_DIM)

corpus_vecs_norm = corpus_vecs.copy()
faiss.normalize_L2(corpus_vecs_norm)  # in-place, same call you used in 02
index.add(corpus_vecs_norm)

print("FAISS index built. ntotal =", index.ntotal)

## 4. ColPali 進階：對 PDF / 文件截圖做 late-interaction 視覺檢索

**WHY 需要 ColPali**：上面 jina-clip 把整張圖壓成「一個向量」。對自然照片夠用，但對**文件頁**（PDF、簡報截圖、表格）這是災難——一頁文件的資訊密度太高，單一向量會把細節抹平。傳統做法是先 OCR 再做文字 RAG，但 OCR 對版面、表格、手寫常出錯。

`vidore/colpali-v1.3` 的思路完全不同：它把**每頁切成多個 patch、每個 patch 各保留一個向量**（multi-vector），query 的每個 token 與所有 patch 算最大相似度再加總——這叫 **late interaction（晚期交互，源自 ColBERT）**。直覺上它在問：「query 的『毛利率』這個詞，在這頁的哪個區塊找得到最像的視覺證據？」**完全免 OCR**。

對照 02 的 cross-encoder rerank：兩者都是「精排」，但 cross-encoder 是純文字、ColPali 是純視覺 multi-vector。

> VRAM 警告：ColPali 基於 PaliGemma，**約需 8 GB+**。若 GPU 不足，可跳過本節（jina-clip 的單向量檢索已能跑完整條 RAG），或改用更小的 `vidore/colSmol-256M`。

In [ ]:
import fitz  # PyMuPDF

def pdf_to_images(pdf_path: str, dpi: int = 150) -> list[Image.Image]:
    """Render each PDF page to a PIL image. ColPali consumes page IMAGES, never text
    -> this is the 'no OCR' part: we feed pixels, not extracted characters."""
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        pages.append(img)
    return pages

# Example usage (uncomment with a real PDF):
# page_images = pdf_to_images("corpus/financial_report.pdf")
# print(f"rendered {len(page_images)} pages")
#
# For a runnable demo without a PDF, reuse our placeholder images as 'pages':
page_images = [load_image(d) for d in CORPUS if d["modality"] == "image"]
page_meta = [d for d in CORPUS if d["modality"] == "image"]
print(f"document pages to index with ColPali: {len(page_images)}")

In [ ]:
from colpali_engine.models import ColPali, ColPaliProcessor

COLPALI_ID = "vidore/colpali-v1.3"

# Lighter alternative for <8 GB GPU:  COLPALI_ID = "vidore/colSmol-256M"
# (ColPali itself does not need 4-bit; if VRAM is tight, prefer colSmol over
#  quantizing colpali, since multi-vector scoring is sensitive to precision.)
colpali = ColPali.from_pretrained(
    COLPALI_ID,
    torch_dtype=DTYPE,
    device_map="auto",
).eval()
colpali_processor = ColPaliProcessor.from_pretrained(COLPALI_ID)

print("ColPali loaded:", COLPALI_ID)

In [ ]:
@torch.inference_mode()
def colpali_embed_pages(images: list[Image.Image]) -> list[torch.Tensor]:
    """Each page -> a MULTI-vector tensor (n_patches, dim), not a single vector.
    This is the structural difference from jina-clip in step 3."""
    batch = colpali_processor.process_images(images).to(colpali.device)
    embs = colpali(**batch)  # (batch, n_patches, dim)
    return list(embs)  # keep per-page multi-vector tensors

page_embeddings = colpali_embed_pages(page_images)
print("per-page multi-vector shapes:", [tuple(e.shape) for e in page_embeddings])

In [ ]:
@torch.inference_mode()
def colpali_search(query_text: str, top_k: int = 3):
    """Late interaction: score query tokens against every page's patches.
    colpali_processor.score_multi_vector implements the MaxSim sum (ColBERT-style).
    """
    q_batch = colpali_processor.process_queries([query_text]).to(colpali.device)
    q_emb = colpali(**q_batch)  # (1, n_query_tokens, dim)
    # scores: (n_queries, n_pages); higher == more relevant page.
    scores = colpali_processor.score_multi_vector(q_emb, page_embeddings)[0]
    top = scores.topk(min(top_k, len(page_embeddings)))
    return [(page_meta[i]["id"], float(s)) for s, i in zip(top.values, top.indices)]

# Example: find the page that visually answers a Chinese query, no OCR involved.
print(colpali_search("哪一頁有貓", top_k=3))

## 5. query（文字或圖）→ 跨模態檢索 top-k 證據

**WHY 統一 retrieve 介面**：02 的 query 永遠是文字。跨模態的威力在於 query 本身也能是圖片（「找跟這張圖最像的文件」）。但下游 FAISS 不該因此分叉——好品味是讓 `retrieve()` 接受任一模態，內部用同一個 encoder 投到同一空間，**搜尋程式碼一行都不改**。這就是消除特殊情況。

In [ ]:
@torch.inference_mode()
def encode_query(query) -> np.ndarray:
    """query can be a str (text) or a PIL.Image. Both map into the SAME jina-clip
    space, so the FAISS search below is modality-agnostic."""
    if isinstance(query, str):
        vec = embed_model.encode_text([query])
    else:  # PIL.Image -> image query
        vec = embed_model.encode_image([query])
    vec = np.asarray(vec, dtype="float32")
    faiss.normalize_L2(vec)  # same normalize as the corpus side
    return vec

def retrieve(query, top_k: int = 3) -> list[dict]:
    """Identical control flow to 02's retrieval: encode -> normalize -> index.search.
    Returns the doc-unit dicts plus their similarity scores."""
    q_vec = encode_query(query)
    scores, idxs = index.search(q_vec, top_k)
    results = []
    for score, i in zip(scores[0].tolist(), idxs[0].tolist()):
        doc = dict(CORPUS[i])
        doc["score"] = score
        results.append(doc)
    return results

In [ ]:
# Text query -> retrieves a MIX of image and text docs from one index.
print("== text query: '曬太陽的貓' ==")
for r in retrieve("曬太陽的貓", top_k=3):
    print(f"  {r['id']:8s} {r['modality']:6s} score={r['score']:.3f} | {r['caption']}")

# Image query -> 'find docs similar to this picture'. Impossible in 02's text-only RAG.
print("\n== image query: corpus/red_car.jpg ==")
for r in retrieve(load_image(CORPUS[1]), top_k=3):
    print(f"  {r['id']:8s} {r['modality']:6s} score={r['score']:.3f} | {r['caption']}")

## 6. 把取回的圖文證據組進 VLM 的多模態 messages

**WHY 這一步最容易踩坑**：02 的 generator prompt 是「一段文字 context + 問題」。VLM 的 prompt 是 `messages`，每則訊息的 `content` 是一個 **list**，圖片用 `{"type": "image"}` 佔位、文字用 `{"type": "text", "text": ...}`。`apply_chat_template` 會把圖片佔位展開成正確數量的 image token 插進序列——**手動拼 `pixel_values` 與 `input_ids` 幾乎必然 token 數對不上**。這正是 03 `vlm_vqa_captioning` 反覆強調的：影像前處理一律交給 processor。

對照 03 你已寫過的：

```python
# 03 的單張圖 VQA
messages = [{"role":"user","content":[{"type":"image"},{"type":"text","text":"這是什麼？"}]}]
```

本節只是把「一張圖」換成「retrieve 回來的 k 張圖 + 它們的中文說明」，並要求模型**標出引用來源**。結構完全相同，只是 content list 更長。

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

VLM_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# 7B VLM in 4-bit NF4 -> ~6-8 GB VRAM. Same quantization recipe as 04-kbits-tuning.
# Lighter alternative for <8 GB:  VLM_ID = "HuggingFaceTB/SmolVLM-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

vlm = AutoModelForImageTextToText.from_pretrained(
    VLM_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
).eval()
vlm_processor = AutoProcessor.from_pretrained(VLM_ID)

print("VLM loaded (4-bit):", VLM_ID)

In [ ]:
def build_messages(question: str, evidence: list[dict]) -> list[dict]:
    """Pack retrieved evidence into multimodal chat messages.

    - image docs  -> {'type':'image','image': PIL}  + a labeled caption text block
    - text docs   -> {'type':'text', ...}
    Each block is prefixed with its source id so the model can CITE it.
    """
    content = [{"type": "text", "text":
                "你是嚴謹的中文助理。只能根據以下提供的證據回答，"
                "每個論點後面用 [來源:id] 標註引用；若證據不足，明說『資料不足』。\n\n證據："}]
    for doc in evidence:
        if doc["modality"] == "image":
            content.append({"type": "text", "text": f"[來源:{doc['id']}] 圖片，說明：{doc['caption']}"})
            content.append({"type": "image", "image": load_image(doc)})
        else:
            content.append({"type": "text", "text": f"[來源:{doc['id']}] 文字：{doc['caption']}"})
    content.append({"type": "text", "text": f"\n問題：{question}"})
    return [{"role": "user", "content": content}]

## 7. VLM 生成有依據的中文回答（含引用來源）

**WHY 用 `apply_chat_template` 而不是自己拼字串**：模板裡藏著 image token 的展開規則、特殊 token、role 標記。Qwen2.5-VL 還需要 `qwen_vl_utils.process_vision_info` 從 messages 抽出實際的影像張量——這對應 02 你把 context 字串塞進 prompt 的步驟，只是這裡有「文字 + 影像」兩條輸入流要同時餵進模型。

In [ ]:
@torch.inference_mode()
def generate_answer(question: str, evidence: list[dict], max_new_tokens: int = 256) -> str:
    messages = build_messages(question, evidence)

    # 1) text side: chat template expands image placeholders into image tokens.
    text = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # 2) vision side: pull the actual image tensors referenced in messages.
    image_inputs, video_inputs = process_vision_info(messages)
    # 3) fuse both streams -> the processor guarantees token counts line up.
    inputs = vlm_processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(vlm.device)

    out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    # Strip the prompt tokens, keep only the freshly generated answer.
    trimmed = out[:, inputs.input_ids.shape[1]:]
    return vlm_processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()

In [ ]:
# Full cross-modal RAG: retrieve -> pack -> generate (the 02 pipeline, upgraded).
def multimodal_rag(question, top_k: int = 3):
    evidence = retrieve(question, top_k=top_k)
    answer = generate_answer(question if isinstance(question, str) else "請描述查詢圖片相關內容", evidence)
    return answer, evidence

QUESTION = "語料裡的貓在做什麼？請說明並標註來源。"
answer, evidence = multimodal_rag(QUESTION, top_k=3)

print("問題：", QUESTION)
print("\n取回證據：")
for e in evidence:
    print(f"  [{e['id']}] {e['modality']} score={e['score']:.3f}")
print("\nVLM 回答：\n", answer)

## 8. 檢索品質評測（Recall@k / NDCG）與生成端幻覺檢查

**WHY 評測分兩段**：RAG 的錯誤有兩個來源——**檢索沒撈到對的證據**（retriever 的鍋），或**證據對了但模型亂講**（generator 幻覺）。02 的 retrieval_bot 只示範了檢索沒做系統評測。這裡補上兩段式檢查，這也是 RAG 上線前的必修。

- **Recall@k**：前 k 筆裡有沒有撈到 ground-truth 相關文件。
- **NDCG@k**：在 Recall 之上加入「排序位置」的權重——相關文件排越前面分數越高。比 02 用過的 MRR 更能反映多個相關文件的情況。
- **幻覺檢查（生成端）**：最務實的做法是檢查回答裡標的 `[來源:id]` 是否真的在我們餵進去的證據集合內。標了不存在的來源 = 明顯幻覺。

In [ ]:
import math

# Ground truth: for each eval query, which doc ids are relevant.
EVAL = [
    {"query": "曬太陽的貓",      "relevant": {"img_001", "txt_001"}},
    {"query": "紅色的跑車",      "relevant": {"img_002", "txt_002"}},
    {"query": "雪山風景",        "relevant": {"img_003"}},
]

def recall_at_k(retrieved_ids: list[str], relevant: set[str], k: int) -> float:
    hit = len(set(retrieved_ids[:k]) & relevant)
    return hit / len(relevant) if relevant else 0.0

def ndcg_at_k(retrieved_ids: list[str], relevant: set[str], k: int) -> float:
    # binary relevance DCG / ideal DCG
    dcg = sum(1.0 / math.log2(rank + 2)
              for rank, did in enumerate(retrieved_ids[:k]) if did in relevant)
    ideal = sum(1.0 / math.log2(rank + 2)
                for rank in range(min(len(relevant), k)))
    return dcg / ideal if ideal > 0 else 0.0

K = 3
recalls, ndcgs = [], []
for item in EVAL:
    got = [r["id"] for r in retrieve(item["query"], top_k=K)]
    recalls.append(recall_at_k(got, item["relevant"], K))
    ndcgs.append(ndcg_at_k(got, item["relevant"], K))
    print(f"{item['query']:>8s} -> {got}  R@{K}={recalls[-1]:.2f}  NDCG@{K}={ndcgs[-1]:.2f}")

print(f"\nmean Recall@{K} = {sum(recalls)/len(recalls):.3f}")
print(f"mean NDCG@{K}   = {sum(ndcgs)/len(ndcgs):.3f}")

In [ ]:
import re

def check_citation_hallucination(answer: str, evidence: list[dict]) -> dict:
    """Cheap, deterministic hallucination probe: every [來源:id] cited by the model
    MUST exist in the evidence we actually provided. Cited-but-absent == hallucination.
    """
    allowed = {d["id"] for d in evidence}
    cited = set(re.findall(r"\[來源:([^\]]+)\]", answer))
    fabricated = cited - allowed
    return {
        "cited": sorted(cited),
        "fabricated_sources": sorted(fabricated),
        "grounded": len(fabricated) == 0 and len(cited) > 0,
    }

report = check_citation_hallucination(answer, evidence)
print(json.dumps(report, ensure_ascii=False, indent=2))
if not report["grounded"]:
    print("\n[警告] 回答缺乏有效引用或出現捏造來源，需人工複查。")

## 9. 快取與索引維護策略（嵌入版本化、增量更新）

**WHY 這是工程而非研究問題**：02 的 demo 每次都重算嵌入、重建索引，跑得動是因為語料小。真實系統語料上萬筆，重算一次嵌入可能要數小時。三條鐵律：

1. **嵌入版本化**：嵌入是「模型 + 前處理」的產物。換 encoder（jina-clip-v1 → v2）或改 normalize 方式，**舊向量與新向量不可混用**——它們不在同一空間。所以每份索引都要綁一個 `embedding_version`，載入時驗證一致才放行。這是「Never break userspace」在 RAG 的對應：別讓不相容的向量悄悄混進來。
2. **增量更新**：新增 doc 只編碼新增的、`index.add` 進去，不要全量重建。`IndexFlatIP` 支援直接 add；刪除則維護一個 id → 是否有效的對照（或定期重建）。
3. **快取**：以 doc 內容的 hash 當 key 快取向量，內容沒變就不重算。

In [ ]:
import hashlib

EMBED_VERSION = f"{EMBED_MODEL_ID}@l2norm"  # bump this whenever encoder/preproc changes

def save_index(index, corpus, dir_path="rag_index"):
    d = Path(dir_path); d.mkdir(exist_ok=True)
    faiss.write_index(index, str(d / "vectors.faiss"))
    meta = {"embedding_version": EMBED_VERSION, "dim": index.d, "corpus": corpus}
    (d / "meta.json").write_text(json.dumps(meta, ensure_ascii=False), encoding="utf-8")

def load_index(dir_path="rag_index"):
    d = Path(dir_path)
    meta = json.loads((d / "meta.json").read_text(encoding="utf-8"))
    # Guard: refuse to serve an index built with a different embedding version.
    if meta["embedding_version"] != EMBED_VERSION:
        raise RuntimeError(
            f"embedding version mismatch: index={meta['embedding_version']} "
            f"vs runtime={EMBED_VERSION}. Re-embed the corpus before serving.")
    return faiss.read_index(str(d / "vectors.faiss")), meta["corpus"]

save_index(index, CORPUS)
_idx, _corpus = load_index()
print("reloaded index ntotal =", _idx.ntotal, "| version verified OK")

In [ ]:
# Content-hash cache + incremental add: only embed what is new, never rebuild.
_vec_cache: dict[str, np.ndarray] = {}

def _doc_hash(doc: dict) -> str:
    key = f"{doc['modality']}|{doc.get('path')}|{doc['caption']}|{EMBED_VERSION}"
    return hashlib.sha256(key.encode("utf-8")).hexdigest()

@torch.inference_mode()
def add_documents(new_docs: list[dict]):
    """Incrementally extend the live index. Cached docs skip re-encoding."""
    new_vecs = []
    for doc in new_docs:
        h = _doc_hash(doc)
        if h not in _vec_cache:
            v = (embed_model.encode_image([load_image(doc)]) if doc["modality"] == "image"
                 else embed_model.encode_text([doc["caption"]]))
            _vec_cache[h] = np.asarray(v, dtype="float32")[0]
        new_vecs.append(_vec_cache[h])
    mat = np.stack(new_vecs).astype("float32")
    faiss.normalize_L2(mat)
    index.add(mat)            # incremental: no full rebuild
    CORPUS.extend(new_docs)   # keep metadata aligned with index rows

add_documents([{"id": "txt_003", "modality": "text", "path": None,
                "caption": "高山針葉林在冬季常被積雪覆蓋"}])
print("after incremental add, ntotal =", index.ntotal, "| corpus =", len(CORPUS))

## 10. 小結、練習題與下一步

### 一句話總結

**RAG = 檢索（嵌入） + 生成（VLM）**。你把 02 的純文字 RAG 拆成「retriever + generator」兩個方塊，然後各自換模態：retriever 換成 jina-clip 圖文共享嵌入（外加 ColPali 文件視覺精排），generator 換成 Qwen2.5-VL。**連線、索引、normalize、top-k、評測心智模型全部沿用**——這就是本模組的整合：02 的視覺嵌入、03 的 VLM 生成、可選 04 的音訊，全部接回你早就會的 RAG 骨架。

### 本模組能力的整合地圖

| 來自 | 在本節扮演的角色 |
| :--- | :--- |
| 01 image_classification（視覺前處理） | 理解影像如何進 encoder |
| 02 clip_retrieval（共享嵌入 + Recall@k） | retriever 的核心 |
| 02 retrieval_bot（FAISS + rerank） | 索引與精排的骨架 |
| 03 vlm_vqa_captioning（image token + chat 模板） | generator 的核心 |
| 04 asr_whisper（可選） | 把語音問題轉文字後接 query（見下方練習 4） |

### 練習題

1. **SigLIP vs CLIP vs jina-clip 對照**：把 step 3 的 encoder 換成 `google/siglip2-base-patch16-224` 與原版 `openai/clip-vit-base-patch32`，用 step 8 的 Recall@k / NDCG 比較三者在中文 query 上的差距。預期：原版 CLIP 中文最差。想想為什麼（提示：文字塔的訓練語料）。
2. **混合檢索 fusion**：對文件頁同時用 jina-clip 單向量檢索與 ColPali late-interaction，做分數融合（如 Reciprocal Rank Fusion），比較單一方法與融合的 NDCG。
3. **更強的幻覺檢查**：step 8 只驗證引用 id 是否存在。進一步用一個 NLI 模型判斷「回答的每句話是否被引用的證據蘊含（entailment）」，把 grounded 從『有引用』升級到『引用真的支持該句』。
4. **接上音訊模態**：用 [`04-asr_whisper`](../04-asr_whisper/whisper_asr.ipynb) 的 Whisper 把一段中文語音轉成文字 query，再餵進 `multimodal_rag()`。驗證「語音 → 文字 → 跨模態檢索 → VLM 生成」整條跑得通。
5. **索引壓縮**：語料放大到上萬筆後，把 `IndexFlatIP` 換成 `IndexIVFFlat` 或 HNSW，量測召回率與延遲的取捨。

### 通往下一份 notebook

本節用的 VLM 是**現成權重**。如果你的領域語料（醫療影像、工業圖、特定中文文件）讓 VLM 答得不好，下一步就是**微調 VLM**：[`06-vlm_finetuning/vlm_lora_finetune.ipynb`](../06-vlm_finetuning/vlm_lora_finetune.ipynb)。它把你在 [`03-PEFT`](../../03-PEFT/README.md) 與 [`04-kbits-tuning`](../../04-kbits-tuning/README.md) 學的 LoRA / 4-bit QLoRA 遷移到 VLM——`target_modules` 多含 projector、image token 要做標籤遮罩，其餘與單模態 QLoRA 一字不差。這就閉合了「單模態微調 → 多模態微調」的整條學習弧線。